# BYOF (Bring Your Own Forecast)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/your_own_forecast.ipynb)

This notebook demonstrates how to wrap your own forecast data and evaluate it with EWB. Uses the WeatherBench2 HRES zarr store as an example. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2019 W European Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9002,
    title="2019 W European Heat Wave (demo)",
    start_date=datetime.datetime(2019, 6, 27),
    end_date=datetime.datetime(2019, 6, 30),
    location=BoundingBoxRegion.create_region(
        latitude_min=45.0,
        latitude_max=51.0,
        longitude_min=0.0,
        longitude_max=6.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
forecast = ewb.ZarrForecast(
    source="gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr",
    name="HRES",
    variables=["surface_air_temperature"],
    variable_mapping=ewb.HRES_metadata_variable_mapping,
    storage_options={"remote_options": {"anon": True}},
)

target = ewb.ERA5(variables=["surface_air_temperature"])

In [ ]:
eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=[
            ewb.metrics.MeanAbsoluteError(
                forecast_variable="surface_air_temperature",
                target_variable="surface_air_temperature",
            ),
            ewb.metrics.MaximumMeanAbsoluteError(
                forecast_variable="surface_air_temperature",
                target_variable="surface_air_temperature",
            ),
        ],
        target=target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()
print(outputs)